# Nigeria Incident Data Analysis — Data Cleaning and Exploratory Analysis

## Project purpose
This notebook documents the data preparation and exploratory analysis for the Nigeria Incident Data Analysis dashboard.

**Input file:** `incidents.csv`  
**Output file:** `cleaned incidents.csv`

The notebook:
1. Loads and inspects the source data.
2. Checks missing values, data types and duplicate records.
3. Converts date and numeric columns to suitable data types.
4. Derives **Year** and **Month** from the incident **Start date**.
5. standardises State labels to remove spelling and formatting duplicates.
6. Explores incident frequency and reported deaths.
7. Exports the cleaned dataset for use in the Streamlit dashboard.

> Important: Year and Month are derived from `Start date` consistently. The earlier notebook reassigned them from `End date`, which could make the time analysis inconsistent.


## 1. Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from collections import Counter

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)


## 2. Load the dataset

In [ ]:
df = pd.read_csv("incidents.csv")

print("Dataset shape:", df.shape)
display(df.head())


## 3. Inspect the dataset

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\\nMissing values:")
display(df.isna().sum().to_frame("Missing Values"))

print("\\nExact duplicate rows:", df.duplicated().sum())


## 4. Clean column values and data types

In [ ]:
# Remove unnecessary whitespace from text fields.
for column in ["Identifier", "Title", "State"]:
    if column in df.columns:
        df[column] = df[column].astype("string").str.strip()

# Convert dates safely. Unparseable values become NaT and are counted below.
df["Start date"] = pd.to_datetime(df["Start date"], errors="coerce")
df["End date"] = pd.to_datetime(df["End date"], errors="coerce")

# Convert deaths to numeric. Missing or invalid death counts are set to zero.
df["Number of deaths"] = pd.to_numeric(
    df["Number of deaths"], errors="coerce"
).fillna(0)

print("Missing Start dates:", df["Start date"].isna().sum())
print("Missing End dates:", df["End date"].isna().sum())
print("Negative death counts:", (df["Number of deaths"] < 0).sum())
print("Updated data types:")
display(df.dtypes.to_frame("Data Type"))


## 5. Derive Year and Month from Start date

The incident start date is used as the consistent reference date for time-based analysis. This avoids mixing start-year values with end-year values.


In [ ]:
df["Year"] = df["Start date"].dt.year.astype("Int64")
df["Month Number"] = df["Start date"].dt.month.astype("Int64")
df["Month"] = df["Start date"].dt.month_name()

display(df[["Start date", "Year", "Month Number", "Month"]].head())


## 6. Standardise State values

The State field contains duplicates caused by inconsistent capitalisation, spelling, spacing, abbreviations and some city/location names. A controlled mapping is used for known variants.

Values that cannot confidently be assigned to a state are retained as **Unclassified** rather than being guessed.


In [ ]:
nigerian_states = [
    "Abia", "Adamawa", "Akwa Ibom", "Anambra", "Bauchi", "Bayelsa",
    "Benue", "Borno", "Cross River", "Delta", "Ebonyi", "Edo", "Ekiti",
    "Enugu", "FCT (Abuja)", "Gombe", "Imo", "Jigawa", "Kaduna", "Kano",
    "Katsina", "Kebbi", "Kogi", "Kwara", "Lagos", "Nasarawa", "Niger",
    "Ogun", "Ondo", "Osun", "Oyo", "Plateau", "Rivers", "Sokoto",
    "Taraba", "Yobe", "Zamfara"
]

aliases = {
    "fct": "FCT (Abuja)",
    "abuja": "FCT (Abuja)",
    "abuja (fct)": "FCT (Abuja)",
    "fct (abuja)": "FCT (Abuja)",
    "fct ( abuja)": "FCT (Abuja)",
    "fct(abuja)": "FCT (Abuja)",
    "(abuja)": "FCT (Abuja)",
    "( abuja)": "FCT (Abuja)",
    "cross rivers": "Cross River",
    "river": "Rivers",
    "port harcourt": "Rivers",
    "bonny": "Rivers",
    "uyo": "Akwa Ibom",
    "niger state": "Niger",
    "rivers (offshore)": "Rivers",
    "bayelsa (offshore)": "Bayelsa",
    "akwa ibom (offshore)": "Akwa Ibom",
    "anmabra": "Anambra",
    "bornon": "Borno",
    "jiagawa": "Jigawa",
    "jigaawa": "Jigawa",
    "jigiwa": "Jigawa",
    "kadauna": "Kaduna",
    "kadun": "Kaduna",
    "katisna": "Katsina",
    "nasaraawa": "Nasarawa",
    "nasarwa": "Nasarawa",
    "zamafara": "Zamfara",
    "katsina 1": "Katsina",
    "katsina 2": "Katsina"
}

state_lookup = {state.casefold(): state for state in nigerian_states}

def standardise_state(value):
    if pd.isna(value) or str(value).strip() == "":
        return "Unclassified"

    raw = re.sub(r"\s+", " ", str(value)).strip()
    key = raw.casefold().strip(" .")

    if key in aliases:
        return aliases[key]
    if key in state_lookup:
        return state_lookup[key]

    # For labels that contain an incident description plus a state name,
    # identify a complete state-name token rather than relying on substring fragments.
    for state in sorted(nigerian_states, key=len, reverse=True):
        pattern = r"(?<![a-z])" + re.escape(state.casefold()) + r"(?![a-z])"
        if re.search(pattern, key):
            return state

    return "Unclassified"

df["State"] = df["State"].apply(standardise_state)

print("Unique State categories after standardisation:", df["State"].nunique())
display(df["State"].value_counts().to_frame("Incident Records"))


## 7. Check state standardisation and remaining unclassified records

In [ ]:
print("State categories:")
print(sorted(df["State"].dropna().unique()))

print("\\nUnclassified records:")
display(df.loc[df["State"] == "Unclassified", ["Identifier", "Title", "State"]])


## 8. Check cleaned data quality

In [ ]:
print("Cleaned dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())

quality_summary = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Unique Values": df.nunique(dropna=True)
})
display(quality_summary)


## 9. Exploratory Data Analysis (EDA)

### 9.1 Number of incidents by year

In [ ]:
incidents_by_year = df["Year"].value_counts().sort_index()
display(incidents_by_year.to_frame("Number of Incidents"))

incidents_by_year.plot(kind="bar", figsize=(9, 5))
plt.title("Number of Incidents by Year")
plt.xlabel("Year")
plt.ylabel("Number of Incidents")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### 9.2 Total reported deaths by year

In [ ]:
deaths_by_year = df.groupby("Year")["Number of deaths"].sum().sort_index()
display(deaths_by_year.to_frame("Reported Deaths"))

deaths_by_year.plot(kind="line", marker="o", figsize=(9, 5))
plt.title("Reported Deaths by Year")
plt.xlabel("Year")
plt.ylabel("Reported Deaths")
plt.tight_layout()
plt.show()


### 9.3 Incidents by month

In [ ]:
month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

incidents_by_month = (
    df["Month"].value_counts().reindex(month_order, fill_value=0)
)
display(incidents_by_month.to_frame("Number of Incidents"))

incidents_by_month.plot(kind="bar", figsize=(10, 5))
plt.title("Number of Incidents by Month")
plt.xlabel("Month")
plt.ylabel("Number of Incidents")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 9.4 Top 10 states by number of incidents

In [ ]:
incidents_by_state = df["State"].value_counts()
top_10_states = incidents_by_state.drop(labels=["Unclassified"], errors="ignore").head(10)
display(top_10_states.to_frame("Number of Incidents"))

top_10_states.sort_values().plot(kind="barh", figsize=(9, 6))
plt.title("Top 10 States by Number of Incidents")
plt.xlabel("Number of Incidents")
plt.ylabel("State")
plt.tight_layout()
plt.show()


### 9.5 Top 10 states by reported deaths

In [ ]:
deaths_by_state = (
    df[df["State"] != "Unclassified"]
    .groupby("State")["Number of deaths"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
display(deaths_by_state.to_frame("Reported Deaths"))

deaths_by_state.sort_values().plot(kind="barh", color="red", figsize=(9, 6))
plt.title("Top 10 States by Reported Deaths")
plt.xlabel("Reported Deaths")
plt.ylabel("State")
plt.tight_layout()
plt.show()


### 9.6 Most frequent incident descriptions

In [ ]:
top_incident_titles = df["Title"].dropna().value_counts().head(10)
display(top_incident_titles.to_frame("Number of Records"))

top_incident_titles.sort_values().plot(kind="barh", figsize=(10, 6))
plt.title("10 Most Frequently Recorded Incident Descriptions")
plt.xlabel("Number of Records")
plt.ylabel("Incident Description")
plt.tight_layout()
plt.show()


### 9.7 Incidents with the highest reported deaths

In [ ]:
top_death_incidents = (
    df.sort_values("Number of deaths", ascending=False)
    [["Identifier", "Title", "State", "Start date", "Number of deaths"]]
    .head(10)
)
display(top_death_incidents)


### 9.8 Average deaths per incident by state

In [ ]:
state_summary = (
    df[df["State"] != "Unclassified"]
    .groupby("State")
    .agg(
        Number_of_Incidents=("Identifier", "count"),
        Total_Deaths=("Number of deaths", "sum")
    )
)
state_summary["Average_Deaths_Per_Incident"] = (
    state_summary["Total_Deaths"] / state_summary["Number_of_Incidents"]
)
display(
    state_summary.sort_values(
        "Average_Deaths_Per_Incident", ascending=False
    ).head(10)
)


## 10. Research questions addressed

1. How has the number of reported incidents changed from year to year?\n2. Which states recorded the highest number of incidents?\n3. Which states recorded the highest number of reported deaths?\n4. Which months recorded the highest number of incidents?\n5. Which incident descriptions appeared most frequently in the dataset?\n6. Which recorded incidents had the highest reported death tolls?\n7. How does the average number of deaths per incident vary across states?\n8. How have reported deaths changed over the years?

## 11. Export the cleaned dataset

In [ ]:
# Keep the dashboard's intended columns in a consistent order.
final_columns = [
    "Identifier", "Title", "Start date", "End date",
    "Number of deaths", "Year", "Month", "State"
]
cleaned_df = df[final_columns].copy()

cleaned_df.to_csv("cleaned_incidents_standardized.csv", index=False, date_format="%Y-%m-%d")

print("Standardised dashboard dataset exported successfully.")
print("Final shape:", cleaned_df.shape)
display(cleaned_df.head())


## 12. Final notes

- `Year` and `Month` are based on `Start date`.
- State names are standardised using the mapping and matching rules documented above.
- Records without a confidently identifiable state are retained as `Unclassified`; review these records manually if the source data can be checked.
- The exported `cleaned incidents.csv` is the dataset to use in the Streamlit dashboard.
